In [5]:
import itk
from typing import Union
from pathlib import Path

### FFT

In [6]:
def fft_processing(input_file: Union[str, Path], output_file: Union[str, Path], window_min: int = 0, window_max: int = 20000, dimension: int = 2) -> None:
    """
    Perform FFT processing on an image.
    
    Args:
        input_file: Path to the input image file
        output_file: Path to save the output image
        window_min: Minimum intensity window value
        window_max: Maximum intensity window value
        dimension: Image dimension (default: 2)
        
    Returns:
        None
    """
    # Define pixel types and image types
    pixel_type = itk.F
    real_image_type = itk.Image[pixel_type, dimension]
    int_pixel_type = itk.US
    int_image_type = itk.Image[int_pixel_type, dimension]
    complex_image_type = itk.Image[itk.complex[pixel_type], dimension]
    
    # Read the input image
    reader = itk.ImageFileReader[real_image_type].New()
    reader.SetFileName(str(input_file))
    reader.Update()
    
    # Apply Forward FFT
    forward_fft_filter = itk.ForwardFFTImageFilter[real_image_type, complex_image_type].New()
    forward_fft_filter.SetInput(reader.GetOutput())
    forward_fft_filter.Update()
    
    # Convert complex to modulus
    complex_to_modulus_filter = itk.ComplexToModulusImageFilter[complex_image_type, real_image_type].New()
    complex_to_modulus_filter.SetInput(forward_fft_filter.GetOutput())
    complex_to_modulus_filter.Update()
    
    # Apply intensity windowing
    windowing_filter = itk.IntensityWindowingImageFilter[real_image_type, int_image_type].New()
    windowing_filter.SetInput(complex_to_modulus_filter.GetOutput())
    windowing_filter.SetWindowMinimum(window_min)
    windowing_filter.SetWindowMaximum(window_max)
    windowing_filter.Update()
    
    # Apply FFT shift
    fft_shift_filter = itk.FFTShiftImageFilter[int_image_type, int_image_type].New()
    fft_shift_filter.SetInput(windowing_filter.GetOutput())
    fft_shift_filter.Update()
    
    # Write the output image
    writer = itk.ImageFileWriter[int_image_type].New()
    writer.SetFileName(str(output_file))
    writer.SetInput(fft_shift_filter.GetOutput())
    writer.Update()

In [7]:
fft_processing(
    "inputs/bar.png",
    "outputs/bar_fft.png",
)

In [8]:
fft_processing(
    "inputs/dot.png",
    "outputs/dot_fft.png",
)

In [9]:
fft_processing(
    "inputs/lehar.png",
    "outputs/lehar_fft.png",
)

In [10]:
fft_processing(
    "inputs/lena.png",
    "outputs/lena_fft.png",
)

### Filtros en dominio de frecuencia

In [11]:
def frequency_domain_filtering(input_file: Union[str, Path], mask_file: Union[str, Path], output_file: Union[str, Path], dimension: int = 2) -> None:
    """
    Perform frequency domain filtering on an image using a mask.
    
    Args:
        input_file: Path to the input image file
        mask_file: Path to the mask image file
        output_file: Path to save the output image
        dimension: Image dimension (default: 2)
        
    Returns:
        None
    """
    # Define pixel types and image types
    pixel_type = itk.F
    real_image_type = itk.Image[pixel_type, dimension]
    char_pixel_type = itk.UC
    char_image_type = itk.Image[char_pixel_type, dimension]
    complex_image_type = itk.Image[itk.complex[pixel_type], dimension]
    
    # Read the input image
    input_reader = itk.ImageFileReader[real_image_type].New()
    input_reader.SetFileName(str(input_file))
    input_reader.Update()
    
    # Read the mask image
    mask_reader = itk.ImageFileReader[char_image_type].New()
    mask_reader.SetFileName(str(mask_file))
    mask_reader.Update()
    
    # Apply Forward FFT to input image
    forward_fft_filter = itk.ForwardFFTImageFilter[real_image_type, complex_image_type].New()
    forward_fft_filter.SetInput(input_reader.GetOutput())
    forward_fft_filter.UpdateOutputInformation()
    
    # Apply FFT shift to mask
    fft_shift_filter = itk.FFTShiftImageFilter[char_image_type, char_image_type].New()
    fft_shift_filter.SetInput(mask_reader.GetOutput())
    fft_shift_filter.Update()
    
    # Apply mask to frequency domain image
    mask_filter = itk.MaskImageFilter[complex_image_type, char_image_type, complex_image_type].New()
    mask_filter.SetInput1(forward_fft_filter.GetOutput())
    mask_filter.SetInput2(fft_shift_filter.GetOutput())
    mask_filter.Update()
    
    # Apply inverse FFT
    inverse_fft_filter = itk.InverseFFTImageFilter[complex_image_type, real_image_type].New()
    inverse_fft_filter.SetInput(mask_filter.GetOutput())
    inverse_fft_filter.Update()
    
    # Calculate min and max values
    min_max_filter = itk.MinimumMaximumImageFilter[real_image_type].New()
    min_max_filter.SetInput(inverse_fft_filter.GetOutput())
    min_max_filter.Update()
    
    min_int = min_max_filter.GetMinimum()
    max_int = min_max_filter.GetMaximum()
    int_range = max_int - min_int
    
    # Apply intensity shift if needed
    int_shift_filter = itk.ShiftScaleImageFilter[real_image_type, real_image_type].New()
    int_shift_filter.SetInput(inverse_fft_filter.GetOutput())
    int_shift_filter.SetShift(-min_int)
    int_shift_filter.Update()
    
    # Rescale intensity if needed
    rescale_filter = itk.RescaleIntensityImageFilter[real_image_type, real_image_type].New()
    if int_range > 255 and min_int < 0:
        rescale_filter.SetInput(int_shift_filter.GetOutput())
    else:
        rescale_filter.SetInput(inverse_fft_filter.GetOutput())
    rescale_filter.SetOutputMinimum(0)
    rescale_filter.SetOutputMaximum(255)
    rescale_filter.Update()
    
    # Cast to char image type
    cast_filter = itk.CastImageFilter[real_image_type, char_image_type].New()
    if int_range > 255:
        cast_filter.SetInput(rescale_filter.GetOutput())
    elif min_int < 0:
        cast_filter.SetInput(int_shift_filter.GetOutput())
    else:
        cast_filter.SetInput(inverse_fft_filter.GetOutput())
    cast_filter.Update()
    
    # Write the output image
    writer = itk.ImageFileWriter[char_image_type].New()
    writer.SetFileName(str(output_file))
    writer.SetInput(cast_filter.GetOutput())
    writer.Update()

In [12]:
frequency_domain_filtering(
    "inputs/lena.png",
    "inputs/invmask1.png",
    "outputs/lena_invmask1.png"
)

In [13]:
frequency_domain_filtering(
    "inputs/lena.png",
    "inputs/invmask2.png",
    "outputs/lena_invmask2.png"
)

In [15]:
frequency_domain_filtering(
    "inputs/lena.png",
    "inputs/invmask3.png",
    "outputs/lena_invmask3.png"
)

In [16]:
frequency_domain_filtering(
    "inputs/lena.png",
    "inputs/invmask4.png",
    "outputs/lena_invmask4.png"
)

In [18]:
frequency_domain_filtering(
    "inputs/lena.png",
    "inputs/mask1.png",
    "outputs/lena_mask1.png"
)

In [19]:
frequency_domain_filtering(
    "inputs/lena.png",
    "inputs/mask2.png",
    "outputs/lena_mask2.png"
)

In [20]:
frequency_domain_filtering(
    "inputs/lena.png",
    "inputs/mask3.png",
    "outputs/lena_mask3.png"
)

In [21]:
frequency_domain_filtering(
    "inputs/lena.png",
    "inputs/mask4.png",
    "outputs/lena_mask4.png"
)